### MODULO 7 - Local AI & Information Extraction con LLM


In questo modulo capiremo come abbandonare approcci tradizionali a favore dell'IA Generativa Locale per interpretare referti PDF disomogenei. Saranno affrontate le seguenti tematiche:
* **Perché l'IA locale?** Privacy dei dati, costi zero e assenza di dipendenza da API esterne.
* **Setup e utilizzo di Ollama:** per far girare modelli open-source (es. Llama 3 o Mistral) in locale su PC/Server aziendali.
* **Tecniche di Structured Data Extraction:** come guidare un LLM tramite Prompt Engineering e schema Pydantic / JSON Mode per estrarre dati strutturati (Parametro, Valore, Unità, Limite) da testi PDF di referti disordinati o non standard.

Prima di iniziare, installiamo le librerie necessarie per Python.

In [ ]:
# Installazione delle librerie per la comunicazione con Ollama e la validazione dati
%pip install ollama pydantic

## 1. Perché l'IA locale e Setup di Ollama
L'integrazione di LLM (Large Language Models) in locale offre enormi vantaggi in ambienti produttivi aziendali:
- **Privacy dei dati**: Nessun dato sensibile lascia i confini del server aziendale.
- **Costi zero**: L'utilizzo di modelli open-source elimina la dipendenza da API esterne a pagamento (es. OpenAI).
- **Assenza di dipendenza da API esterne**: Garantisce il funzionamento offline o all'interno di reti aziendali isolate.

Per questo corso useremo **Ollama**, uno strumento formidabile per far girare modelli open-source (es. Llama 3 o Mistral) in locale.

*Nota:* Assicurati di aver installato Ollama sul tuo sistema operativo:
- Vai sul sito ufficiale ollama.com.
- Clicca su Download e scarica la versione per il tuo SO.
- Avvia il file scaricato e completa l'installazione.
- Avvia tramite terminale con `ollama run llama3`.

In [ ]:
import ollama

# Test rapido di connessione al modello locale
try:
    response = ollama.chat(model='llama3', messages=[
        {
            'role': 'user',
            'content': 'Ciao, rispondi con una sola frase: sei pronto a estrarre dati?'
        }
    ])
    print("--- RISPOSTA OLLAMA ---")
    print(response['message']['content'])
except Exception as e:
    print("⚠️ Errore di connessione a Ollama! Assicurati che l'applicazione desktop o il servizio Ollama sia in esecuzione sul tuo PC/Server.")

## 2. Structured Data Extraction (Prompt Engineering & JSON Mode)
Il nostro obiettivo è utilizzare il Prompt Engineering per prendere i testi provenienti da referti disordinati o non standard ed estrarre informazioni chiave: Parametro, Valore, Unità, Limite.

I modelli moderni supportano il **JSON Mode**, che costringe l'LLM a rispondere esclusivamente con un oggetto JSON valido, perfettamente integrabile in un DataFrame Pandas.

In [ ]:
import json

# Simuliamo un referto in formato testo libero disordinato estratto da un PDF
testo_referto_grezzo = """
Risultati del prelievo Acque di Scarico - 30/07/2026.
Si rileva un pH pari a 7.3, rientrante nel limite di 6.5-9.5.
L'analisi dei Nitrati evidenzia un picco di 45.5 mg/L, ma comunque sotto il limite massimo di 50.
I cloruri non sono stati rilevati in questo campione.
"""

# Tecnica di Prompt Engineering per l'Information Extraction
prompt = f"""
Estrai i parametri chimici dal seguente referto testuale.
Restituisci l'output ESCLUSIVAMENTE in formato JSON con la seguente struttura:
Una chiave principale 'risultati' contenente una lista di dizionari.
Ogni dizionario deve avere le chiavi: 'Parametro', 'Valore', 'Unita', 'Limite'.
Se un dato non è indicato o non rilevato, assegna il valore null.

Referto grezzo:
{testo_referto_grezzo}
"""

try:
    # Inviamo il prompt a Ollama abilitando la modalità JSON
    risposta_json = ollama.chat(
        model='llama3',
        messages=[{'role': 'user', 'content': prompt}],
        format='json'  # 👈 Questo parametro forza l'output a essere puro JSON!
    )

    # Conversione da stringa a dizionario Python tramite il modulo json
    dati_strutturati = json.loads(risposta_json['message']['content'])

    print("--- DATI ESTRATTI (Dizionario Python) ---")
    print(json.dumps(dati_strutturati, indent=4, ensure_ascii=False))

except Exception as e:
    print(f"⚠️ Si è verificato un errore: {e}")

## 3. Schema Pydantic: Forzare la Struttura e Validare
Oltre al semplice Prompt Engineering, per avere la certezza matematica che l'LLM rispetti i tipi di dato (float, stringhe) possiamo appoggiarci a **Pydantic**.
Ci permette di definire uno schema rigoroso che potremo usare sia per spiegare la struttura all'LLM, sia per validarne l'output finale in Python.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

# Definiamo la struttura esatta che vogliamo ottenere dal referto
class Misurazione(BaseModel):
    parametro: str = Field(description="Il nome del parametro chimico (es. Nitrati, pH)")
    valore: Optional[float] = Field(description="Il valore numerico. Null se non rilevato")
    unita: Optional[str] = Field(description="Unità di misura (es. mg/L)")
    limite: Optional[str] = Field(description="Il limite massimo consentito")

class Referto(BaseModel):
    risultati: List[Misurazione]

print("Ecco come Pydantic 'spiega' lo schema al modello in automatico:")
print(json.dumps(Referto.model_json_schema(), indent=2))

# Esempio di validazione a valle
# referto_validato = Referto(**dati_strutturati)
# print(referto_validato.model_dump())

# Esercitazione (1h): Modulo `ai_extractor.py`

**Obiettivo:** Creazione di un modulo Python (simulato qui in cella) che prende il testo grezzo estratto da un PDF "difficile", lo invia al modello Ollama locale e restituisce un dizionario/JSON perfettamente conformato agli oggetti definiti.

## Il tuo compito:
1. Crea una funzione `estrai_dati_pdf(testo)` che racchiuda la logica vista sopra.
2. Scrivi un prompt eccellente che istruisca il modello a mappare Parametro, Valore, Unita e Limite.
3. Assicurati di usare `format='json'` nella chiamata a Ollama.
4. Fai testare la funzione passando la variabile stringa `pdf_difficile_text` fornita di seguito.
5. Restituisci il dizionario pulito e stampalo a video!

Buon lavoro!

In [ ]:
import json
import ollama

# --- SETUP: TESTO DI INPUT ---
pdf_difficile_text = """
===============================
LABORATORI CAMPANIA ANALISI
Referto n. 45001 del 12/03/2026
===============================
Dati del campione: PNT-SUD
Osservazioni visive: Torbidità non presente.
Riscontriamo tracce di Ferro pari a 0,15 milligrammi per litro. Lim. normativo: 0.2
Escherichia Coli zero. Limite zero (UFC in 100ml).
Concentrazione di Solfati nella norma (24 mg/l). Limite massimo 250.
"""

# Inizia il tuo codice qui sotto 👇

def estrai_dati_pdf(testo):
    # 1. Definisci il tuo prompt

    # 2. Invia la richiesta a Ollama

    # 3. Trasforma il JSON restituito in Dict e ritornalo
    pass


# 4. Esegui la funzione e stampa il risultato
# ris = estrai_dati_pdf(pdf_difficile_text)
# print(ris)

<details>
<summary><b>👀 Clicca qui per vedere la soluzione dell'esercitazione</b></summary>

```python
def estrai_dati_pdf(testo):
    prompt_system = f"""
    Sei un assistente AI specializzato nell'estrazione dati di referti chimici.
    Analizza il testo seguente e restituisci un JSON contenente una chiave 'risultati' con una lista di oggetti.
    Ogni oggetto deve corrispondere ai parametri analizzati e contenere: 'Parametro', 'Valore' (come numero se possibile), 'Unita', 'Limite'.
    Se un dato non è presente metti null.
    
    Testo da analizzare:
    {testo}
    """
    
    try:
        # Chiamata al modello con json_mode
        response = ollama.chat(
            model='llama3',
            messages=[{'role': 'user', 'content': prompt_system}],
            format='json'
        )
        
        # Trasformazione
        risultato_dizionario = json.loads(response['message']['content'])
        return risultato_dizionario
        
    except Exception as e:
        print(f"Errore nell'estrazione: {e}")
        return None

# Test in live
dati = estrai_dati_pdf(pdf_difficile_text)
print(json.dumps(dati, indent=4, ensure_ascii=False))
```
</details>